In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to summarize a dataframe with timestamp column and name column
def summarize(df, ts_col='timestamp', id_col='cmdb_id', name_col=None, name_label='names', limit=20):
    rows = int(df.shape[0])
    if rows == 0:
        min_ts = max_ts = None
        min_dt = max_dt = None
    else:
        min_ts = int(df[ts_col].min())
        max_ts = int(df[ts_col].max())
        min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
        max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
    cmdb_vals = list(pd.Series(df[id_col].unique()).astype(str))
    cmdb_count = len(cmdb_vals)
    cmdb_display = cmdb_vals[:limit]
    cmdb_display_str = ', '.join(cmdb_display)
    if name_col is not None and name_col in df.columns:
        name_vals = list(pd.Series(df[name_col].unique()).astype(str))
        name_count = len(name_vals)
        name_display = name_vals[:limit]
        name_display_str = ', '.join(name_display)
    else:
        name_count = 0
        name_display_str = ''
    return {
        'rows': rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'min_dt_Asia/Shanghai': min_dt,
        'max_dt_Asia/Shanghai': max_dt,
        'cmdb_count': cmdb_count,
        'cmdb_ids_display': cmdb_display_str,
        f'{name_label}_count': name_count,
        f'{name_label}_display': name_display_str
    }

# Summaries
metric_sum = summarize(metric_df, ts_col='timestamp', id_col='cmdb_id', name_col='kpi_name', name_label='kpi_names')
trace_sum = summarize(trace_df, ts_col='timestamp', id_col='cmdb_id', name_col='trace_name', name_label='trace_names')
log_sum = summarize(log_df, ts_col='timestamp', id_col='cmdb_id', name_col='log_name', name_label='log_names')
error_sum = summarize(error_logs_df, ts_col='timestamp', id_col='cmdb_id', name_col='message', name_label='messages')

# Build a compact inventory DataFrame
inventory = pd.DataFrame([
    {
        'file': 'metric.csv',
        'rows': metric_sum['rows'],
        'min_ts': metric_sum['min_timestamp'],
        'max_ts': metric_sum['max_timestamp'],
        'min_dt_Asia/Shanghai': metric_sum['min_dt_Asia/Shanghai'],
        'max_dt_Asia/Shanghai': metric_sum['max_dt_Asia/Shanghai'],
        'distinct_cmdb_ids_count': metric_sum['cmdb_count'],
        'distinct_cmdb_ids_show': metric_sum['cmdb_ids_display'],
        'distinct_kpi_count': metric_sum['kpi_names_count'],
        'distinct_kpi_show': metric_sum['kpi_names_display']
    },
    {
        'file': 'trace.csv',
        'rows': trace_sum['rows'],
        'min_ts': trace_sum['min_timestamp'],
        'max_ts': trace_sum['max_timestamp'],
        'min_dt_Asia/Shanghai': trace_sum['min_dt_Asia/Shanghai'],
        'max_dt_Asia/Shanghai': trace_sum['max_dt_Asia/Shanghai'],
        'distinct_cmdb_ids_count': trace_sum['cmdb_count'],
        'distinct_cmdb_ids_show': trace_sum['cmdb_ids_display'],
        'distinct_trace_count': trace_sum['trace_names_count'],
        'distinct_trace_show': trace_sum['trace_names_display']
    },
    {
        'file': 'log.csv',
        'rows': log_sum['rows'],
        'min_ts': log_sum['min_timestamp'],
        'max_ts': log_sum['max_timestamp'],
        'min_dt_Asia/Shanghai': log_sum['min_dt_Asia/Shanghai'],
        'max_dt_Asia/Shanghai': log_sum['max_dt_Asia/Shanghai'],
        'distinct_cmdb_ids_count': log_sum['cmdb_count'],
        'distinct_cmdb_ids_show': log_sum['cmdb_ids_display'],
        'distinct_log_count': log_sum['log_names_count'],
        'distinct_log_show': log_sum['log_names_display']
    },
    {
        'file': 'error_logs.csv',
        'rows': error_sum['rows'],
        'min_ts': error_sum['min_timestamp'],
        'max_ts': error_sum['max_timestamp'],
        'min_dt_Asia/Shanghai': error_sum['min_dt_Asia/Shanghai'],
        'max_dt_Asia/Shanghai': error_sum['max_dt_Asia/Shanghai'],
        'distinct_cmdb_ids_count': error_sum['cmdb_count'],
        'distinct_cmdb_ids_show': error_sum['cmdb_ids_display'],
        'distinct_message_count': error_sum['messages_count'],
        'distinct_message_show': error_sum['messages_display']
    }
])

# Compact top 5 rows for error_logs (select only requested columns)
error_top5 = error_logs_df[['timestamp', 'cmdb_id', 'message']].head(5)

# For compactness convert datetime columns to ISO strings
inventory['min_dt_Asia/Shanghai'] = inventory['min_dt_Asia/Shanghai'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
inventory['max_dt_Asia/Shanghai'] = inventory['max_dt_Asia/Shanghai'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Display inventory (compact) and top 5 error logs
inventory, error_top5

```
Out[1]:
```
```python
# Build a plain-English summary based on the previously computed `inventory` and `error_top5`
row = inventory.set_index('file')

metric_row = row.loc['metric.csv']
trace_row = row.loc['trace.csv']
log_row = row.loc['log.csv']
error_row = row.loc['error_logs.csv']

summary = (
    "Telemetry inventory summary:\n\n"
    f"- metric.csv: {int(metric_row['rows'])} rows; time range {metric_row['min_dt_Asia/Shanghai']} to {metric_row['max_dt_Asia/Shanghai']}; "
    f"{int(metric_row['distinct_cmdb_ids_count'])} distinct cmdb_id(s) (examples: {metric_row['distinct_cmdb_ids_show']}); "
    f"{int(metric_row['distinct_kpi_count'])} distinct KPI names (showing up to 20): {metric_row['distinct_kpi_show']}\n\n"
    f"- trace.csv: {int(trace_row['rows'])} rows; time range {trace_row['min_dt_Asia/Shanghai']} to {trace_row['max_dt_Asia/Shanghai']}; "
    f"{int(trace_row['distinct_cmdb_ids_count'])} distinct cmdb_id(s) (examples: {trace_row['distinct_cmdb_ids_show']}); "
    f"{int(trace_row['distinct_trace_count'])} distinct trace names (showing up to 20): {trace_row['distinct_trace_show']}\n\n"
    f"- log.csv: {int(log_row['rows'])} rows; time range {log_row['min_dt_Asia/Shanghai']} to {log_row['max_dt_Asia/Shanghai']}; "
    f"{int(log_row['distinct_cmdb_ids_count'])} distinct cmdb_id(s) (examples: {log_row['distinct_cmdb_ids_show']}); "
    f"{int(log_row['distinct_log_count'])} distinct log names (showing up to 20): {log_row['distinct_log_show']}\n\n"
    f"- error_logs.csv: {int(error_row['rows'])} rows (total error messages = {int(error_row['rows'])}). "
    "No error log rows available to show (top 5 is empty).\n\n"
    "All times are in Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows        min_ts        max_ts       min_dt_Asia/Shanghai       max_dt_Asia/Shanghai  distinct_cmdb_ids_count                             distinct_cmdb_ids_show  distinct_kpi_count                                  distinct_kpi_show  distinct_trace_count                                distinct_trace_show  distinct_log_count                                 distinct_log_show  distinct_message_count  \
0      metric.csv  27144  1.615534e+09  1.615536e+09  2021-03-12T15:30:00+08:00  2021-03-12T16:00:00+08:00                       29  IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redi...               347.0  JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM-...                   NaN                                                NaN                 NaN                                               NaN                     NaN   
1       trace.csv   7068  1.615534e+09  1.615536e+09  2021-03-12T15:30:00+08:00  2021-03-12T15:59:00+08:00                       12  IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, To...                 NaN                                                NaN                  72.0  trace.self.duration_mean, trace.self.duration_...                 NaN                                               NaN                     NaN   
2         log.csv    532  1.615534e+09  1.615536e+09  2021-03-12T15:30:00+08:00  2021-03-12T16:00:00+08:00                        7  Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat0...                 NaN                                                NaN                   NaN                                                NaN                 3.0  log.error_count, log.row_count, log.gc_pause_max                     NaN   
3  error_logs.csv      0           NaN           NaN                       None                       None                        0                                                                    NaN                                                NaN                   NaN                                                NaN                 NaN                                               NaN                     0.0   

  distinct_message_show  
0                   NaN  
1                   NaN  
2                   NaN  
3                        , Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import re
import pandas as pd

# Reuse existing metric_df loaded earlier in the session
# 1) Top 20 kpi_name by row count
top_kpis = (
    metric_df
    .groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Match kpi_name by substrings (case-insensitive) and return kpi_name, count, distinct cmdb_ids (comma-separated, limit 20)
substrs = ['cpu','mem','memory','jvm','heap','disk','io','net','network','latency','load','ss']
pattern = '|'.join(re.escape(s) for s in substrs)

mask = metric_df['kpi_name'].astype(str).str.contains(pattern, case=False, na=False)

matched = metric_df[mask].copy()

matched_kpis = (
    matched
    .groupby('kpi_name')
    .agg(
        count=('kpi_name', 'size'),
        cmdb_ids=('cmdb_id', lambda s: ','.join(sorted(pd.unique(s.astype(str)))[:20]))
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(200)  # limit total returned rows to 200
    .reset_index(drop=True)
)

# Display compact results
top_kpis, matched_kpis

```
Out[2]:
```
```python
# Build a plain-English summary using existing `top_kpis` and `matched_kpis` variables
top_list = [
    f"{i+1}. {row.kpi_name} — {int(row.count)}"
    for i, row in top_kpis.reset_index(drop=True).iterrows()
]
top_section = "Top 20 KPI names by row count:\n" + "\n".join(top_list)

total_matched = matched_kpis.shape[0]
matched_head = matched_kpis.head(20)
matched_lines = []
for _, r in matched_head.iterrows():
    cmdbs = r.cmdb_ids
    cmdb_count = 0 if pd.isna(cmdbs) or cmdbs=='' else len(str(cmdbs).split(','))
    matched_lines.append(f"- {r.kpi_name} — {int(r.count)} rows — cmdb_ids (showing up to 20): {cmdbs} ({cmdb_count} distinct)")

matched_section = (
    f"\nKPI names matching substrings (cpu, mem, memory, jvm, heap, disk, io, net, network, latency, load, ss):\n"
    f"Total distinct matched KPI names: {total_matched}\n"
    + "\n".join(matched_lines)
)

summary = (
    top_section
    + "\n\n"
    + matched_section
    + "\n\nAll outputs are compact; cmdb_id lists are limited to the first 20 entries shown where applicable."
)

# Display the summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count
0       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    363
1                  OSLinux-OSLinux_ZABBIX_Host_Uptime    363
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    363
3      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    357
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    357
5                              OSLinux-CPU_CPU_CPUWio    346
6                             OSLinux-CPU_CPU_CPULoad    345
7                         OSLinux-CPU_CPU_CPUidleutil    345
8                         OSLinux-CPU_CPU_CPUUserTime    344
9   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    344
10                         OSLinux-CPU_CPU_CPUSysTime    344
11                                            app.mrt    341
12                                            app.cnt    341
13                                             app.sr    341
14                                             app.rr    341
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    290
16       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    276
17                         OSLinux-CPU_CPU_CPUCpuUtil    272
18             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    271
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    270,                                               kpi_name  count                                           cmdb_ids
0           OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    363  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...
1        OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    363  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...
2       OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    357  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...
3         OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    357  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...
4                               OSLinux-CPU_CPU_CPUWio    346  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...
..                                                 ...    ...                                                ...
151  Container-DOCKER_CONTAINER_d27123361435--bcou-...      2                                           dockerB2
152  Container-DOCKER_CONTAINER_d27123361435--bcou-...      1                                           dockerB2
153  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...      1                                           dockerA1
154  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...      1                                           dockerA1
155  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...      1                                           dockerA1

[156 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and matched_kpis from session
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# List of KPI names to analyze (from matched_kpis)
kpi_list = matched_kpis['kpi_name'].astype(str).unique().tolist()

# 1) Global aggregation (full-series) by cmdb_id and kpi_name: P95, P5, total_points
grp = metric_df[metric_df['kpi_name'].isin(kpi_list)].groupby(['cmdb_id', 'kpi_name'])

group_stats = grp['value'].agg(
    total_points='size',
    global_p95=lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan,
    global_p5 = lambda s: np.nanpercentile(s.dropna(), 5) if s.dropna().size>0 else np.nan
).reset_index()

# 2) Filter rows within incident window and compute incident metrics
# Define incident window in Asia/Shanghai and convert to UTC timestamps for comparison
start_dt = pd.Timestamp('2021-03-12 15:30:00').tz_localize(tz)
end_dt   = pd.Timestamp('2021-03-12 16:00:00').tz_localize(tz)

# Convert metric_df timestamps to timezone-aware datetimes in Asia/Shanghai
metric_df['_dt_utc8'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

incident_df = metric_df[
    (metric_df['kpi_name'].isin(kpi_list)) &
    (metric_df['_dt_utc8'] >= start_dt) &
    (metric_df['_dt_utc8'] <= end_dt)
].copy()

# Merge thresholds into incident_df
incident_df = incident_df.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Compute per-group incident-window aggregates
incident_aggs = incident_df.groupby(['cmdb_id','kpi_name']).agg(
    incident_window_points = ('value','size'),
    min_incident_value = ('value','min'),
    max_incident_value = ('value','max')
).reset_index()

# Earliest anomaly timestamps for high and low separately
# High anomalies: value > global_p95
high_mask = incident_df['value'] > incident_df['global_p95']
high_df = incident_df[high_mask].copy()
if not high_df.empty:
    high_first = high_df.groupby(['cmdb_id','kpi_name'])['_dt_utc8'].min().reset_index().rename(columns={'_dt_utc8':'earliest_high_ts'})
else:
    high_first = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_high_ts'])

# Low anomalies: value < global_p5
low_mask = incident_df['value'] < incident_df['global_p5']
low_df = incident_df[low_mask].copy()
if not low_df.empty:
    low_first = low_df.groupby(['cmdb_id','kpi_name'])['_dt_utc8'].min().reset_index().rename(columns={'_dt_utc8':'earliest_low_ts'})
else:
    low_first = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_low_ts'])

# Combine everything
result = group_stats.merge(incident_aggs, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(high_first, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(low_first, on=['cmdb_id','kpi_name'], how='left')

# Replace NaNs for incident counts with 0
result['incident_window_points'] = result['incident_window_points'].fillna(0).astype(int)

# 3) Determine anomaly existence, compute severity for high and low, choose the more severe if both
# Compute severity candidates
result['severity_high'] = 0.0
result['severity_low'] = 0.0

# For severity_high need max_incident_value; for severity_low need min_incident_value
# Fill min/max incident values with NaN where no incident rows
# Compute severity only when relevant
mask_has_max = result['max_incident_value'].notna() & result['global_p95'].notna()
result.loc[mask_has_max, 'severity_high'] = ((result.loc[mask_has_max, 'max_incident_value'] - result.loc[mask_has_max, 'global_p95']) / result.loc[mask_has_max, 'global_p95']).clip(lower=0)

mask_has_min = result['min_incident_value'].notna() & result['global_p5'].notna()
result.loc[mask_has_min, 'severity_low'] = ((result.loc[mask_has_min, 'global_p5'] - result.loc[mask_has_min, 'min_incident_value']) / result.loc[mask_has_min, 'global_p5']).clip(lower=0)

# Determine chosen anomaly type and severity
def pick_anomaly(row):
    sh, sl = row['severity_high'], row['severity_low']
    if (sh <= 0) and (sl <= 0):
        return pd.Series([np.nan, np.nan, None])  # no anomaly
    if sh >= sl:
        # choose high
        ts = row['earliest_high_ts']
        return pd.Series(['high', sh, ts])
    else:
        ts = row['earliest_low_ts']
        return pd.Series(['low', sl, ts])

result[['anomaly_type','severity','earliest_anomaly_ts']] = result.apply(pick_anomaly, axis=1)

# Keep only groups that have at least one incident-window anomaly (severity > 0 and incident_window_points>0)
final = result[
    (result['severity'].notna()) &
    (result['severity'] > 0) &
    (result['incident_window_points'] > 0)
].copy()

# Format earliest_anomaly_ts to Asia/Shanghai ISO strings
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()
final['earliest_anomaly_timestamp_Asia/Shanghai'] = final['earliest_anomaly_ts'].apply(fmt_ts)

# Prepare final output columns
out_cols = [
    'cmdb_id','kpi_name','total_points','incident_window_points',
    'global_p95','global_p5','min_incident_value','max_incident_value',
    'earliest_anomaly_timestamp_Asia/Shanghai','anomaly_type','severity'
]
final_out = final[out_cols].copy()

# Sort by severity desc and limit to top 20
final_out_sorted = final_out.sort_values('severity', ascending=False).reset_index(drop=True).head(20)

# Keep output compact: show numeric rounding for readability
final_out_sorted['global_p95'] = final_out_sorted['global_p95'].round(6)
final_out_sorted['global_p5'] = final_out_sorted['global_p5'].round(6)
final_out_sorted['min_incident_value'] = final_out_sorted['min_incident_value'].round(6)
final_out_sorted['max_incident_value'] = final_out_sorted['max_incident_value'].round(6)
final_out_sorted['severity'] = final_out_sorted['severity'].round(6)

final_out_sorted

```
Out[3]:
```
Summary of anomalies (analysis over full series; incident window 2021-03-12 15:30:00 to 2021-03-12 16:00:00 Asia/Shanghai). Groups shown are those with at least one incident-window anomaly, sorted by severity (top 20):

1) IG01 | OSLinux-CPU_CPU_CPUWio — high — severity 3.3336 — earliest anomaly 2021-03-12T15:47:00+08:00 — points in window 30 — incident min/max 0.016636 / 1.114700 — global P95/P5 = 0.257225 / 0.020791

2) Redis02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite — high — severity 1.7785 — earliest 2021-03-12T15:30:00+08:00 — points 15 — incident min/max 5.733300 / 143.333300 — global P95/P5 = 51.586657 / 5.826680

3) Redis02 | OSLinux-CPU_CPU_CPUWio — high — severity 1.6611 — earliest 2021-03-12T15:47:00+08:00 — points 29 — incident min/max 0.004177 / 0.204800 — global P95/P5 = 0.076960 / 0.004189

4) IG02 | OSLinux-CPU_CPU_CPUWio — high — severity 1.3712 — earliest 2021-03-12T15:40:00+08:00 — points 31 — incident min/max 0.016600 / 0.847951 — global P95/P5 = 0.357600 / 0.020790

5) apache02 | OSLinux-CPU_CPU_CPUWio — high — severity 1.2286 — earliest 2021-03-12T15:39:00+08:00 — points 29 — incident min/max 0.008300 / 0.528088 — global P95/P5 = 0.236960 / 0.012469

6) Redis02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps — high — severity 1.1782 — earliest 2021-03-12T15:30:00+08:00 — points 15 — incident min/max 0.450000 / 3.666700 — global P95/P5 = 1.683343 / 0.450000

7) Tomcat02 | OSLinux-CPU_CPU_CPUWio — high — severity 1.0113 — earliest 2021-03-12T15:40:00+08:00 — points 16 — incident min/max 0.016700 / 1.111100 — global P95/P5 = 0.552425 / 0.019775

8) Tomcat04 | Tomcat-Threads_7441-"http-nio-8003"_CurrentThreads — low — severity 1.0000 — earliest 2021-03-12T15:53:00+08:00 — points 31 — incident min/max 0.000000 / 14.000000 — global P95/P5 = 11.000000 / 1.000000

9) Redis01 | OSLinux-CPU_CPU-0_SingleCpuidle — low — severity 1.0000 — earliest 2021-03-12T15:41:00+08:00 — points 15 — incident min/max 0.000000 / 96.182100 — global P95/P5 = 96.180280 / 20.247570

10) Tomcat03 | OSLinux-CPU_CPU-2_SingleCpuidle — low — severity 1.0000 — earliest 2021-03-12T15:54:00+08:00 — points 9 — incident min/max 0.000000 / 97.537400 — global P95/P5 = 97.524120 / 0.485280

11) Tomcat02 | Tomcat-Threads_7441-"http-nio-8003"_CurrentThreads — low — severity 1.0000 — earliest 2021-03-12T15:59:00+08:00 — points 31 — incident min/max 0.000000 / 19.000000 — global P95/P5 = 13.000000 / 1.000000

12) IG02 | OSLinux-CPU_CPU-0_SingleCpuidle — low — severity 0.9894 — earliest 2021-03-12T16:00:00+08:00 — points 16 — incident min/max 0.149500 / 98.518400 — global P95/P5 = 98.507000 / 14.072609

13) Tomcat04 | OSLinux-CPU_CPU-0_SingleCpuUtil — high — severity 0.8498 — earliest 2021-03-12T15:47:00+08:00 — points 11 — incident min/max 2.062500 / 45.921200 — global P95/P5 = 24.825400 / 2.194800

14) Redis02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead — high — severity 0.8452 — earliest 2021-03-12T15:32:00+08:00 — points 11 — incident min/max 5.000000 / 143.000000 — global P95/P5 = 77.500000 / 5.000000

15) apache01 | OSLinux-CPU_CPU-0_SingleCpuUtil — high — severity 0.7739 — earliest 2021-03-12T15:30:00+08:00 — points 10 — incident min/max 0.564100 / 45.204100 — global P95/P5 = 25.483210 / 0.788560

16) IG02 | OSLinux-CPU_CPU-0_SingleCpuUtil — high — severity 0.7705 — earliest 2021-03-12T15:59:00+08:00 — points 10 — incident min/max 1.496800 / 81.286400 — global P95/P5 = 45.911630 / 1.511425

17) MG01 | OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil — low — severity 0.7525 — earliest 2021-03-12T15:41:00+08:00 — points 9 — incident min/max 0.000500 / 0.007200 — global P95/P5 = 0.007080 / 0.002020

18) apache01 | OSLinux-CPU_CPU-3_SingleCpuUtil — low — severity 0.7385 — earliest 2021-03-12T15:40:00+08:00 — points 11 — incident min/max 3.984700 / 100.000000 — global P95/P5 = 100.000000 / 15.239850

19) apache01 | OSLinux-CPU_CPU-2_SingleCpuidle — low — severity 0.7120 — earliest 2021-03-12T15:41:00+08:00 — points 15 — incident min/max 1.310300 / 99.468200 — global P95/P5 = 99.445030 / 4.550390

20) apache01 | OSLinux-CPU_CPU-2_SingleCpuUtil — high — severity 0.6958 — earliest 2021-03-12T15:40:00+08:00 — points 11 — incident min/max 0.531800 / 49.617900 — global P95/P5 = 29.260050 / 0.548350

Notes:
- Severity is computed relative to the global P95/P5 (high anomalies: (max_incident - P95)/P95; low anomalies: (P5 - min_incident)/P5).
- All times are Asia/Shanghai (UTC+8). The table is limited to the top 20 groups by severity and contains only aggregated metrics (no raw time series).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  incident_window_points  global_p95  global_p5  min_incident_value  max_incident_value earliest_anomaly_timestamp_Asia/Shanghai anomaly_type  severity
0       IG01                             OSLinux-CPU_CPU_CPUWio            30                      30    0.257225   0.020791            0.016636            1.114700                2021-03-12T15:47:00+08:00         high  3.333563
1    Redis02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite            15                      15   51.586657   5.826680            5.733300          143.333300                2021-03-12T15:30:00+08:00         high  1.778496
2    Redis02                             OSLinux-CPU_CPU_CPUWio            29                      29    0.076960   0.004189            0.004177            0.204800                2021-03-12T15:47:00+08:00         high  1.661123
3       IG02                             OSLinux-CPU_CPU_CPUWio            31                      31    0.357600   0.020790            0.016600            0.847951                2021-03-12T15:40:00+08:00         high  1.371228
4   apache02                             OSLinux-CPU_CPU_CPUWio            29                      29    0.236960   0.012469            0.008300            0.528088                2021-03-12T15:39:00+08:00         high  1.228596
5    Redis02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps            15                      15    1.683343   0.450000            0.450000            3.666700                2021-03-12T15:30:00+08:00         high  1.178225
6   Tomcat02                             OSLinux-CPU_CPU_CPUWio            16                      16    0.552425   0.019775            0.016700            1.111100                2021-03-12T15:40:00+08:00         high  1.011314
7   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31                      31   11.500000   1.000000            0.000000           14.000000                2021-03-12T15:53:00+08:00          low  1.000000
8    Redis01                    OSLinux-CPU_CPU-0_SingleCpuidle            15                      15   96.180280  20.247570            0.000000           96.182100                2021-03-12T15:41:00+08:00          low  1.000000
9   Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle             9                       9   97.524120   0.485280            0.000000           97.537400                2021-03-12T15:54:00+08:00          low  1.000000
10  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31                      31   13.000000   1.000000            0.000000           19.000000                2021-03-12T15:59:00+08:00          low  1.000000
11      IG02                    OSLinux-CPU_CPU-0_SingleCpuidle            16                      16   98.507000  14.072609            0.149500           98.518400                2021-03-12T16:00:00+08:00          low  0.989377
12  Tomcat04                    OSLinux-CPU_CPU-0_SingleCpuUtil            11                      11   24.825400   2.194800            2.062500           45.921200                2021-03-12T15:47:00+08:00         high  0.849767
13   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...            11                      11   77.500000   5.000000            5.000000          143.000000                2021-03-12T15:32:00+08:00         high  0.845161
14  apache01                    OSLinux-CPU_CPU-0_SingleCpuUtil            10                      10   25.483210   0.788560            0.564100           45.204100                2021-03-12T15:30:00+08:00         high  0.773878
15      IG02                    OSLinux-CPU_CPU-0_SingleCpuUtil            10                      10   45.911630   1.511425            1.496800           81.286400                2021-03-12T15:59:00+08:00         high  0.770497
16      MG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil             9                       9    0.007080   0.002020            0.000500            0.007200                2021-03-12T15:41:00+08:00          low  0.752475
17  apache01                    OSLinux-CPU_CPU-3_SingleCpuUtil            11                      11  100.000000  15.239850            3.984700          100.000000                2021-03-12T15:40:00+08:00          low  0.738534
18  apache01                    OSLinux-CPU_CPU-2_SingleCpuidle            15                      15   99.445030   4.550390            1.310300           99.468200                2021-03-12T15:41:00+08:00          low  0.712047
19  apache01                    OSLinux-CPU_CPU-2_SingleCpuUtil            11                      11   29.260050   0.548350            0.531800           49.617900                2021-03-12T15:40:00+08:00         high  0.695756```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df from session
tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids
cmdb_list = ['IG01','Redis02','IG02','apache02','Tomcat02','Tomcat04']

# Ensure trace timestamps as timezone-aware Asia/Shanghai datetimes
trace_df['_dt_utc8'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter full-series to relevant cmdb_ids
trace_sub = trace_df[trace_df['cmdb_id'].isin(cmdb_list)].copy()

# 1) Global aggregation by cmdb_id and trace_name: total_points and global_p95
grp = trace_sub.groupby(['cmdb_id','trace_name'])['value']
group_stats = grp.agg(
    total_points = 'size',
    global_p95 = lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan
).reset_index()

# 2) Incident window filtering
start_dt = pd.Timestamp('2021-03-12 15:30:00').tz_localize(tz)
end_dt   = pd.Timestamp('2021-03-12 16:00:00').tz_localize(tz)

incident_mask = (trace_sub['_dt_utc8'] >= start_dt) & (trace_sub['_dt_utc8'] <= end_dt)
incident_df = trace_sub[incident_mask].copy()

# Aggregate incident-window metrics per group
incident_aggs = incident_df.groupby(['cmdb_id','trace_name']).agg(
    incident_window_points = ('value','size'),
    max_incident_value = ('value','max')
).reset_index()

# Earliest timestamp in window where value > global_p95 (need to merge global_p95)
incident_with_thresh = incident_df.merge(group_stats, on=['cmdb_id','trace_name'], how='left')

# Filter anomalies > global_p95 (respecting NaN)
anomaly_mask = incident_with_thresh['value'] > incident_with_thresh['global_p95']
anomaly_df = incident_with_thresh[anomaly_mask].copy()

if not anomaly_df.empty:
    earliest_anomaly = anomaly_df.groupby(['cmdb_id','trace_name'])['_dt_utc8'].min().reset_index().rename(columns={'_dt_utc8':'earliest_anomaly_ts'})
else:
    earliest_anomaly = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_ts'])

# Combine group_stats + incident_aggs + earliest_anomaly
result = group_stats.merge(incident_aggs, on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_anomaly, on=['cmdb_id','trace_name'], how='left')

# Fill missing incident counts and max values for groups with no data in window
result['incident_window_points'] = result['incident_window_points'].fillna(0).astype(int)
result['max_incident_value'] = result['max_incident_value'].where(result['max_incident_value'].notna(), np.nan)

# Format earliest anomaly timestamp to Asia/Shanghai ISO strings
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

result['earliest_anomaly_timestamp_Asia/Shanghai'] = result['earliest_anomaly_ts'].apply(fmt_ts)

# Select and order columns as requested
out_cols = [
    'cmdb_id','trace_name','total_points','global_p95',
    'incident_window_points','max_incident_value','earliest_anomaly_timestamp_Asia/Shanghai'
]
final_out = result[out_cols].copy()

# Round numeric columns for compactness
final_out['global_p95'] = final_out['global_p95'].round(6)
final_out['max_incident_value'] = final_out['max_incident_value'].round(6)

# Sort by max_incident_value desc and limit to top 50
final_out_sorted = final_out.sort_values('max_incident_value', ascending=False).reset_index(drop=True).head(50)

# Display compact result
final_out_sorted

```
Out[4]:
```
# Plain-English summary of the trace analysis for specified cmdb_ids (Asia/Shanghai timezone)
top_n = 10
cmdb_list = ['IG01','Redis02','IG02','apache02','Tomcat02','Tomcat04']

# `final_out_sorted` was computed previously and contains the requested columns and sorting.
# Use it to build a compact textual summary showing the top N rows by max_incident_value.
rows = final_out_sorted.head(top_n)

summary_lines = []
summary_lines.append("Trace analysis summary for cmdb_ids: " + ", ".join(cmdb_list) + ".")
summary_lines.append(f"Total distinct (cmdb_id, trace_name) groups analyzed: {final_out_sorted.shape[0]}.")
summary_lines.append("Top groups by incident-window max value (showing up to {}):".format(top_n))

for i, r in rows.reset_index(drop=True).iterrows():
    summary_lines.append(
        f"{i+1}. {r.cmdb_id} | {r.trace_name} — total_points={int(r.total_points)}, "
        f"global_P95={float(r.global_p95):.3f}, window_points={int(r.incident_window_points)}, "
        f"max_in_window={float(r.max_incident_value):.3f}, "
        f"earliest_above_P95={r.earliest_anomaly_timestamp_Asia/Shanghai}"
    )

summary_lines.append("All times are Asia/Shanghai (UTC+8). No raw time series are shown; values are aggregated as requested.")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  total_points   global_p95  incident_window_points  max_incident_value earliest_anomaly_timestamp_Asia/Shanghai
0   Tomcat02             trace.self.row_count            30  3809.750000                      30         4023.000000                2021-03-12T15:31:00+08:00
1   Tomcat04             trace.self.row_count            30  3747.300000                      30         3950.000000                2021-03-12T15:32:00+08:00
2       IG02             trace.self.row_count            30  2227.100000                      30         2350.000000                2021-03-12T15:31:00+08:00
3       IG01             trace.self.row_count            30  2276.000000                      30         2296.000000                2021-03-12T15:31:00+08:00
4   Tomcat02          trace.to_MG02.row_count            30   444.100000                      30          466.000000                2021-03-12T15:38:00+08:00
..       ...                              ...           ...          ...                     ...                 ...                                      ...
45      IG02    trace.from_root.duration_mean             6     4.612250                       6            5.803000                2021-03-12T15:33:00+08:00
46  Tomcat04    trace.from_IG02.duration_mean            30     1.476983                      30            2.562424                2021-03-12T15:30:00+08:00
47      IG02  trace.to_Tomcat04.duration_mean            30     1.476983                      30            2.562424                2021-03-12T15:30:00+08:00
48      IG02         trace.self.duration_mean            30     1.624582                      30            2.318366                2021-03-12T15:30:00+08:00
49  Tomcat02    trace.from_IG02.duration_mean            30     1.606638                      30            2.305934                2021-03-12T15:30:00+08:00

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df and its timezone-aware datetime exist (reusing from session if present)
if '_dt_utc8' not in trace_df.columns:
    trace_df['_dt_utc8'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Find groups where trace_name contains 'redis' or 'to_redis' or 'from_redis' (case-insensitive)
pattern = r'redis|to_redis|from_redis'
mask = trace_df['trace_name'].astype(str).str.contains(pattern, case=False, na=False)
trace_redis = trace_df[mask].copy()

# 2) Global aggregation (full series) by cmdb_id and trace_name: total_points and global_p95
group_stats = (
    trace_redis
    .groupby(['cmdb_id','trace_name'])['value']
    .agg(
        total_points = 'size',
        global_p95 = lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 3) Incident window filtering and aggregations
start_dt = pd.Timestamp('2021-03-12 15:30:00').tz_localize(tz)
end_dt   = pd.Timestamp('2021-03-12 16:00:00').tz_localize(tz)

incident_mask = (trace_redis['_dt_utc8'] >= start_dt) & (trace_redis['_dt_utc8'] <= end_dt)
incident_df = trace_redis[incident_mask].copy()

incident_aggs = (
    incident_df
    .groupby(['cmdb_id','trace_name'])
    .agg(
        incident_window_points = ('value','size'),
        max_incident_value = ('value','max')
    )
    .reset_index()
)

# 4) Earliest timestamp in window where value > global_p95
incident_with_thresh = incident_df.merge(group_stats, on=['cmdb_id','trace_name'], how='left')
anomaly_mask = incident_with_thresh['value'] > incident_with_thresh['global_p95']
anomaly_df = incident_with_thresh[anomaly_mask].copy()

if not anomaly_df.empty:
    earliest_anomaly = (
        anomaly_df
        .groupby(['cmdb_id','trace_name'])['_dt_utc8']
        .min()
        .reset_index()
        .rename(columns={'_dt_utc8':'earliest_anomaly_ts'})
    )
else:
    earliest_anomaly = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_ts'])

# 5) Combine results
result = group_stats.merge(incident_aggs, on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_anomaly, on=['cmdb_id','trace_name'], how='left')

# Fill missing incident counts with 0; keep max_incident_value as NaN when no window data
result['incident_window_points'] = result['incident_window_points'].fillna(0).astype(int)

# Format earliest anomaly timestamp to Asia/Shanghai ISO strings
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

result['earliest_anomaly_timestamp_Asia/Shanghai'] = result['earliest_anomaly_ts'].apply(fmt_ts)

# Select and order columns as requested
out_cols = [
    'cmdb_id','trace_name','total_points','global_p95',
    'incident_window_points','max_incident_value','earliest_anomaly_timestamp_Asia/Shanghai'
]
redis_final = result[out_cols].copy()

# Round numeric columns for compactness
redis_final['global_p95'] = redis_final['global_p95'].round(6)
redis_final['max_incident_value'] = redis_final['max_incident_value'].round(6)

# 6) Sort by max_incident_value desc and limit to top 20
redis_final_sorted = redis_final.sort_values('max_incident_value', ascending=False).reset_index(drop=True).head(20)

# Display compact result variable
redis_final_sorted

```
Out[5]:
```
summary = (
    "Result: No (cmdb_id, trace_name) groups matched the substrings 'redis', 'to_redis', or 'from_redis' "
    "in trace.csv (case-insensitive). Therefore the requested aggregated table (total_points, global_P95, "
    "incident_window_points, max_incident_value, earliest timestamp > P95 in window) is empty for the "
    "incident window 2021-03-12 15:30:00+08:00 to 2021-03-12 16:00:00+08:00. All times are Asia/Shanghai (UTC+8)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, total_points, global_p95, incident_window_points, max_incident_value, earliest_anomaly_timestamp_Asia/Shanghai]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and trace_df from the session
tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime columns exist
if '_dt_utc8' not in metric_df.columns:
    metric_df['_dt_utc8'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if '_dt_utc8' not in trace_df.columns:
    trace_df['_dt_utc8'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window bounds (Asia/Shanghai)
start_dt = pd.Timestamp('2021-03-12 15:30:00').tz_localize(tz)
end_dt   = pd.Timestamp('2021-03-12 16:00:00').tz_localize(tz)

# -------------------------
# Part 1: Redis02 disk I/O KPIs
# -------------------------
redis_kpis = [
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps"
]

# Filter full-series for Redis02 and these KPIs
redis_full = metric_df[(metric_df['cmdb_id']=='Redis02') & (metric_df['kpi_name'].isin(redis_kpis))].copy()

# Compute global P95 per KPI (full series)
redis_p95 = (
    redis_full
    .groupby('kpi_name')['value']
    .agg(global_p95=lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
    .reset_index()
)

# Prepare incident-window subset (for Redis02, those KPIs)
redis_window = redis_full[(redis_full['_dt_utc8'] >= start_dt) & (redis_full['_dt_utc8'] <= end_dt)].copy()
# Ensure sorted by timestamp within each KPI for consecutive detection
redis_window = redis_window.sort_values(['kpi_name','_dt_utc8'])

# For each KPI compute incident_window_points and detect consecutive segments where value > global_p95
rows = []
for kpi in redis_kpis:
    p95_row = redis_p95[redis_p95['kpi_name']==kpi]
    global_p95 = float(p95_row['global_p95'].iloc[0]) if not p95_row.empty else np.nan
    df_k = redis_window[redis_window['kpi_name']==kpi].copy()
    incident_points = int(df_k.shape[0])
    if incident_points == 0:
        n_segments = 0
        earliest_start = None
        earliest_len = 0
        earliest_max = np.nan
    else:
        # boolean mask of anomalies within window
        mask = df_k['value'] > global_p95
        if mask.any():
            # identify consecutive True segments: compute segment ids
            seg_id = (mask != mask.shift(1)).cumsum()
            df_k = df_k.assign(_anomaly=mask, _seg=seg_id)
            anomaly_segs = df_k[df_k['_anomaly']].groupby('_seg').agg(
                start_ts = ('_dt_utc8','min'),
                length_points = ('value','size'),
                max_value = ('value','max')
            ).reset_index(drop=True).sort_values('start_ts')
            n_segments = anomaly_segs.shape[0]
            # earliest segment
            earliest = anomaly_segs.iloc[0]
            earliest_start = earliest['start_ts'].tz_convert(tz).isoformat()
            earliest_len = int(earliest['length_points'])
            earliest_max = float(earliest['max_value'])
        else:
            n_segments = 0
            earliest_start = None
            earliest_len = 0
            earliest_max = np.nan

    rows.append({
        'kpi_name': kpi,
        'global_p95': round(global_p95,6) if not np.isnan(global_p95) else np.nan,
        'incident_window_points': incident_points,
        'number_of_consecutive_anomaly_segments': n_segments,
        'earliest_segment_start_Asia/Shanghai': earliest_start,
        'earliest_segment_length_minutes': earliest_len,
        'earliest_segment_max_value': round(earliest_max,6) if not np.isnan(earliest_max) else np.nan
    })

redis_check_df = pd.DataFrame(rows)[[
    'kpi_name','global_p95','incident_window_points',
    'number_of_consecutive_anomaly_segments','earliest_segment_start_Asia/Shanghai',
    'earliest_segment_length_minutes','earliest_segment_max_value'
]]

# -------------------------
# Part 2: trace.self.duration_mean across all cmdb_ids
# -------------------------
trace_name = 'trace.self.duration_mean'
trace_full = trace_df[trace_df['trace_name']==trace_name].copy()

# Full-series global P95 per cmdb_id and total_points
trace_stats = (
    trace_full
    .groupby('cmdb_id')['value']
    .agg(
        total_points = 'size',
        global_p95 = lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan
    )
    .reset_index()
)

# Incident window subset for this trace_name
trace_window = trace_full[(trace_full['_dt_utc8'] >= start_dt) & (trace_full['_dt_utc8'] <= end_dt)].copy()

# Incident window aggregations per cmdb_id
trace_inc_aggs = (
    trace_window
    .groupby('cmdb_id')
    .agg(
        incident_window_points = ('value','size'),
        max_incident_value = ('value','max')
    )
    .reset_index()
)

# Earliest timestamp in window where value > global_p95
# Merge global_p95 into trace_window to evaluate
trace_window_thresh = trace_window.merge(trace_stats[['cmdb_id','global_p95']], on='cmdb_id', how='left')
mask_anom = trace_window_thresh['value'] > trace_window_thresh['global_p95']
if mask_anom.any():
    earliest_ts = (
        trace_window_thresh[mask_anom]
        .groupby('cmdb_id')['_dt_utc8']
        .min()
        .reset_index()
        .rename(columns={'_dt_utc8':'earliest_above_p95_ts'})
    )
else:
    earliest_ts = pd.DataFrame(columns=['cmdb_id','earliest_above_p95_ts'])

# Combine stats
trace_result = trace_stats.merge(trace_inc_aggs, on='cmdb_id', how='left')
trace_result = trace_result.merge(earliest_ts, on='cmdb_id', how='left')

# Fill missing incident counts with 0
trace_result['incident_window_points'] = trace_result['incident_window_points'].fillna(0).astype(int)
# Format earliest timestamps
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

trace_result['earliest_above_P95_timestamp_Asia/Shanghai'] = trace_result['earliest_above_p95_ts'].apply(fmt_ts)
# Round numeric columns and prepare final columns
trace_result['global_p95'] = trace_result['global_p95'].round(6)
trace_result['max_incident_value'] = trace_result['max_incident_value'].round(6)
final_trace_df = trace_result[[
    'cmdb_id','total_points','global_p95','incident_window_points',
    'earliest_above_P95_timestamp_Asia/Shanghai','max_incident_value'
]]

# Sort by earliest_above_P95_timestamp ascending (None go to bottom), limit top 20
final_trace_df['__sort_ts'] = pd.to_datetime(final_trace_df['earliest_above_P95_timestamp_Asia/Shanghai']).fillna(pd.Timestamp('2100-01-01'))
final_trace_df_sorted = final_trace_df.sort_values('__sort_ts').drop(columns='__sort_ts').reset_index(drop=True).head(20)

# Final outputs (compact)
redis_check_df, final_trace_df_sorted

```
Out[6]:
```
Part 1 — Redis02 disk I/O KPIs (full-series P95; incident window 2021-03-12 15:30–16:00 Asia/Shanghai)
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite
  - global P95: 51.586657
  - incident_window_points: 15
  - number_of_consecutive_anomaly_segments (value > P95): 1
  - earliest anomaly segment start: 2021-03-12T15:30:00+08:00
  - earliest segment length (minutes / consecutive points): 1
  - max value in that earliest segment: 143.3333
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
  - global P95: 0.000000
  - incident_window_points: 15
  - number_of_consecutive_anomaly_segments: 0 (no values > P95 in window)
  - earliest segment start: None
  - earliest segment length: 0
  - earliest segment max value: N/A
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps
  - global P95: 1.683343
  - incident_window_points: 15
  - number_of_consecutive_anomaly_segments: 1
  - earliest anomaly segment start: 2021-03-12T15:30:00+08:00
  - earliest segment length: 1
  - max value in that earliest segment: 3.6667

Part 2 — trace.self.duration_mean (per cmdb_id; full-series global P95; incident-window earliest exceedance and max)
- All reported cmdb_ids had 30 total points and 30 points in the incident window (2021-03-12 15:30–16:00).
- Earliest timestamp where value > global P95 (if any) and max value in window for top entries (earliest first):
  1. IG01 — global_P95=1.609039 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=2.156360
  2. IG02 — global_P95=1.624582 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=2.318366
  3. MG02 — global_P95=1.233880 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=1.349345
  4. Tomcat01 — global_P95=0.212466 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.324567
  5. Tomcat02 — global_P95=0.235857 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.334521
  6. Tomcat03 — global_P95=0.252986 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.332622
  7. Tomcat04 — global_P95=0.232288 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.333567
  8. dockerA1 — global_P95=0.202599 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.265988
  9. dockerA2 — global_P95=0.146474 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.152287
 10. dockerB2 — global_P95=0.271831 — earliest_above_P95=2021-03-12T15:30:00+08:00 — max_in_window=0.288743
 11. MG01 — global_P95=1.282230 — earliest_above_P95=2021-03-12T15:32:00+08:00 — max_in_window=1.711072
 12. dockerB1 — global_P95=0.176451 — earliest_above_P95=2021-03-12T15:32:00+08:00 — max_in_window=0.191788

Notes:
- Times shown are Asia/Shanghai (UTC+8).
- Outputs are aggregated (no raw time series). Only the requested KPIs and trace groups were evaluated and results are bounded as requested.

The original code execution output of IPython Kernel is also provided below for reference:

(                                           kpi_name  global_p95  incident_window_points  number_of_consecutive_anomaly_segments earliest_segment_start_Asia/Shanghai  earliest_segment_length_minutes  earliest_segment_max_value
0  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite   51.586657                      15                                       1            2021-03-12T15:30:00+08:00                                1                    143.3333
1   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.000000                      15                                       0                                 None                                0                         NaN
2   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps    1.683343                      15                                       1            2021-03-12T15:30:00+08:00                                1                      3.6667,      cmdb_id  total_points  global_p95  incident_window_points earliest_above_P95_timestamp_Asia/Shanghai  max_incident_value
0       IG01            30    1.609039                      30                  2021-03-12T15:30:00+08:00            2.156360
1       IG02            30    1.624582                      30                  2021-03-12T15:30:00+08:00            2.318366
2       MG02            30    1.233880                      30                  2021-03-12T15:30:00+08:00            1.349345
3   Tomcat01            30    0.212466                      30                  2021-03-12T15:30:00+08:00            0.324567
4   Tomcat02            30    0.235857                      30                  2021-03-12T15:30:00+08:00            0.334521
5   Tomcat03            30    0.252986                      30                  2021-03-12T15:30:00+08:00            0.332622
6   Tomcat04            30    0.232288                      30                  2021-03-12T15:30:00+08:00            0.333567
7   dockerA1            30    0.202599                      30                  2021-03-12T15:30:00+08:00            0.265988
8   dockerA2            30    0.146474                      30                  2021-03-12T15:30:00+08:00            0.152287
9   dockerB2            30    0.271831                      30                  2021-03-12T15:30:00+08:00            0.288743
10      MG01            30    1.282230                      30                  2021-03-12T15:32:00+08:00            1.711072
11  dockerB1            30    0.176451                      30                  2021-03-12T15:32:00+08:00            0.191788)```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids
cmdbs = ['Redis02','IG01','IG02','Tomcat02','Tomcat04','Tomcat01','Tomcat03','MG02']

# Ensure log_df exists in session and has timezone-aware datetime column
if '_dt_utc8' not in log_df.columns:
    log_df['_dt_utc8'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to target cmdb_ids
log_sub = log_df[log_df['cmdb_id'].isin(cmdbs)].copy()

# Incident window bounds (Asia/Shanghai)
start_dt = pd.Timestamp('2021-03-12 15:30:00').tz_localize(tz)
end_dt   = pd.Timestamp('2021-03-12 16:00:00').tz_localize(tz)

# 1) Global thresholds (computed on full-series BEFORE filtering)
group_stats = (
    log_sub
    .groupby(['cmdb_id','log_name'])['value']
    .agg(
        total_points = 'size',
        global_p95 = lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan,
        global_p5  = lambda s: np.nanpercentile(s.dropna(), 5)  if s.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) Incident-window subset and aggregations
log_window = log_sub[(log_sub['_dt_utc8'] >= start_dt) & (log_sub['_dt_utc8'] <= end_dt)].copy()

incident_aggs = (
    log_window
    .groupby(['cmdb_id','log_name'])
    .agg(
        incident_window_points = ('value','size'),
        min_incident_value = ('value','min'),
        max_incident_value = ('value','max')
    )
    .reset_index()
)

# 3) Earliest anomaly timestamps within window for high (>P95) and low (<P5)
# Merge thresholds to windowed data
window_with_thresh = log_window.merge(group_stats, on=['cmdb_id','log_name'], how='left')

# High anomalies
high_mask = window_with_thresh['value'] > window_with_thresh['global_p95']
high_df = window_with_thresh[high_mask].copy()
if not high_df.empty:
    earliest_high = high_df.groupby(['cmdb_id','log_name'])['_dt_utc8'].min().reset_index().rename(columns={'_dt_utc8':'earliest_high_ts'})
else:
    earliest_high = pd.DataFrame(columns=['cmdb_id','log_name','earliest_high_ts'])

# Low anomalies
low_mask = window_with_thresh['value'] < window_with_thresh['global_p5']
low_df = window_with_thresh[low_mask].copy()
if not low_df.empty:
    earliest_low = low_df.groupby(['cmdb_id','log_name'])['_dt_utc8'].min().reset_index().rename(columns={'_dt_utc8':'earliest_low_ts'})
else:
    earliest_low = pd.DataFrame(columns=['cmdb_id','log_name','earliest_low_ts'])

# 4) Combine stats
res = group_stats.merge(incident_aggs, on=['cmdb_id','log_name'], how='left')
res = res.merge(earliest_high, on=['cmdb_id','log_name'], how='left')
res = res.merge(earliest_low, on=['cmdb_id','log_name'], how='left')

# Fill missing incident counts with 0
res['incident_window_points'] = res['incident_window_points'].fillna(0).astype(int)

# 5) Compute severity for high and low (avoid division by zero)
res['severity_high'] = np.nan
res['severity_low'] = np.nan

mask_h = res['max_incident_value'].notna() & res['global_p95'].notna() & (res['global_p95'] != 0)
res.loc[mask_h, 'severity_high'] = ((res.loc[mask_h, 'max_incident_value'] - res.loc[mask_h, 'global_p95']) / res.loc[mask_h, 'global_p95']).clip(lower=0)

mask_l = res['min_incident_value'].notna() & res['global_p5'].notna() & (res['global_p5'] != 0)
res.loc[mask_l, 'severity_low'] = ((res.loc[mask_l, 'global_p5'] - res.loc[mask_l, 'min_incident_value']) / res.loc[mask_l, 'global_p5']).clip(lower=0)

# Replace NaN severities with 0 for comparison
res['severity_high'] = res['severity_high'].fillna(0)
res['severity_low'] = res['severity_low'].fillna(0)

# Determine chosen anomaly type and severity (choose the larger)
def choose_type(row):
    if row['incident_window_points'] == 0:
        return pd.Series([None, 0.0, None])
    if (row['severity_high'] <= 0) and (row['severity_low'] <= 0):
        return pd.Series([None, 0.0, None])
    if row['severity_high'] >= row['severity_low']:
        ts = row['earliest_high_ts']
        return pd.Series(['high', row['severity_high'], ts])
    else:
        ts = row['earliest_low_ts']
        return pd.Series(['low', row['severity_low'], ts])

res[['anomaly_type','severity','earliest_anomaly_ts']] = res.apply(choose_type, axis=1)

# Keep only groups that have at least one incident-window anomaly (severity>0)
final = res[(res['severity'] > 0) & (res['incident_window_points'] > 0)].copy()

# Format earliest timestamp to Asia/Shanghai ISO strings
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.Timestamp(x).tz_convert(tz).isoformat()

final['earliest_anomaly_timestamp_Asia/Shanghai'] = final['earliest_anomaly_ts'].apply(fmt_ts)

# Prepare final columns and rounding
out_cols = [
    'cmdb_id','log_name','total_points','global_p95','global_p5',
    'incident_window_points','min_incident_value','max_incident_value',
    'earliest_anomaly_timestamp_Asia/Shanghai','anomaly_type','severity'
]
final_out = final[out_cols].copy()
final_out['global_p95'] = final_out['global_p95'].round(6)
final_out['global_p5'] = final_out['global_p5'].round(6)
final_out['min_incident_value'] = final_out['min_incident_value'].round(6)
final_out['max_incident_value'] = final_out['max_incident_value'].round(6)
final_out['severity'] = final_out['severity'].round(6)

# Sort by severity desc and limit to top 20
final_out_sorted = final_out.sort_values('severity', ascending=False).reset_index(drop=True).head(20)

# Display compact result
final_out_sorted

```
Out[7]:
```
Summary (log.csv; incident window 2021-03-12 15:30–16:00 Asia/Shanghai).  
Only the following (cmdb_id, log_name) groups had incident-window anomalies (severity computed relative to global P95/P5 as described):

1. Tomcat02 — log.row_count — low — severity 0.989189 — earliest anomaly 2021-03-12T15:49:00+08:00  
   total_points=31; global_P95=572.0; global_P5=370.0; window_points=31; min=4.00; max=587.00

2. Tomcat01 — log.row_count — low — severity 0.986979 — earliest anomaly 2021-03-12T15:58:00+08:00  
   total_points=31; global_P95=563.5; global_P5=384.0; window_points=31; min=5.00; max=587.00

3. Tomcat03 — log.row_count — low — severity 0.986877 — earliest anomaly 2021-03-12T15:48:00+08:00  
   total_points=31; global_P95=567.5; global_P5=381.0; window_points=31; min=5.00; max=604.00

4. Tomcat04 — log.row_count — low — severity 0.968170 — earliest anomaly 2021-03-12T15:58:00+08:00  
   total_points=31; global_P95=555.5; global_P5=377.0; window_points=31; min=12.00; max=573.00

5. Tomcat02 — log.gc_pause_max — low — severity 0.487179 — earliest anomaly 2021-03-12T15:56:00+08:00  
   total_points=30; global_P95=0.09; global_P5=0.039; window_points=30; min=0.02; max=0.11

6. Tomcat03 — log.gc_pause_max — low — severity 0.365079 — earliest anomaly 2021-03-12T15:45:00+08:00  
   total_points=24; global_P95=0.09; global_P5=0.0315; window_points=24; min=0.02; max=0.10

7. Tomcat01 — log.gc_pause_max — low — severity 0.333333 — earliest anomaly 2021-03-12T15:32:00+08:00  
   total_points=24; global_P95=0.10; global_P5=0.03; window_points=24; min=0.02; max=0.10

8. Tomcat04 — log.gc_pause_max — high — severity 0.116751 — earliest anomaly 2021-03-12T15:44:00+08:00  
   total_points=24; global_P95=0.0985; global_P5=0.02; window_points=24; min=0.02; max=0.11

Notes:
- Only groups with at least one incident-window anomaly are shown (sorted by severity, top 20 limited — here 8 groups).  
- Severity = (max_in_window - global_P95)/global_P95 for high anomalies, or (global_P5 - min_in_window)/global_P5 for low anomalies.  
- All timestamps are Asia/Shanghai (UTC+8). Outputs are aggregated; no raw time series returned.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  total_points  global_p95  global_p5  incident_window_points  min_incident_value  max_incident_value earliest_anomaly_timestamp_Asia/Shanghai anomaly_type  severity
0  Tomcat02     log.row_count            31    572.0000   370.0000                      31                4.00              587.00                2021-03-12T15:49:00+08:00          low  0.989189
1  Tomcat01     log.row_count            31    563.5000   384.0000                      31                5.00              587.00                2021-03-12T15:58:00+08:00          low  0.986979
2  Tomcat03     log.row_count            31    567.5000   381.0000                      31                5.00              604.00                2021-03-12T15:48:00+08:00          low  0.986877
3  Tomcat04     log.row_count            31    555.5000   377.0000                      31               12.00              573.00                2021-03-12T15:58:00+08:00          low  0.968170
4  Tomcat02  log.gc_pause_max            30      0.0900     0.0390                      30                0.02                0.11                2021-03-12T15:56:00+08:00          low  0.487179
5  Tomcat03  log.gc_pause_max            24      0.0900     0.0315                      24                0.02                0.10                2021-03-12T15:45:00+08:00          low  0.365079
6  Tomcat01  log.gc_pause_max            24      0.1000     0.0300                      24                0.02                0.10                2021-03-12T15:32:00+08:00          low  0.333333
7  Tomcat04  log.gc_pause_max            24      0.0985     0.0200                      24                0.02                0.11                2021-03-12T15:44:00+08:00         high  0.116751```
```